# EquipeGo 2 — Piloto 01: ETA en CPU y GPU

**Objetivo:** mismo cálculo y mismos datos en la CPU y GPU de Google Colab. Basado en la fórmula de `Dispatcher.cpp` (distancias obtenidas previamente por Dijkstra); **no reemplaza el despachador original**.

Comienza activando **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.


In [ ]:
import subprocess
print('=== GPU disponible ===')
subprocess.run(['nvidia-smi'], check=True)
print('=== Compilador CUDA ===')
subprocess.run(['nvcc', '--version'], check=True)
print('=== CPU (resumen) ===')
subprocess.run('lscpu | head -n 16', shell=True, check=False)


## Descargar el repositorio independiente

Se usa `EquipeGo-2` y nunca se hace push al `EquipeGo` del Bloque 1. Si decidimos mantenerlo privado, ajustaremos la autenticación de forma segura.


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/EquipeGo-2')
url = 'https://github.com/Leandro0210/EquipeGo-2.git'
if not repo.exists():
    subprocess.run(['git', 'clone', url, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
modulo = repo / 'bloque2_cuda'
assert (modulo / '01_eta_cpu_gpu.cu').exists(), 'Falta subir el modulo bloque2_cuda al repo nuevo'
print('Modulo localizado en:', modulo)


## Compilar con nvcc

Compilamos el mismo archivo `.cu` que incluye la referencia CPU y el kernel CUDA.


In [ ]:
binario = modulo / 'eta_benchmark'
subprocess.run(['nvcc', '-O3', '-std=c++17', str(modulo/'01_eta_cpu_gpu.cu'), '-o', str(binario)], check=True)
print('Compilación completada:', binario)


## Verificar primero un tamaño pequeño

`VALIDACION,OK` indica que CPU y GPU produjeron valores compatibles; los tiempos aún deben interpretarse.


In [ ]:
subprocess.run([str(binario), '1000', '256', '15'], check=True)


## Experimento de escala y tamaño de bloque

Usamos una misma semilla, mediana de 15 repeticiones, entradas de 1.000 a 1.000.000 y varios tamaños de bloque. El tiempo GPU total incluye copias y lanzamiento, pero excluye `cudaMalloc`/`cudaFree`.


In [ ]:
import pandas as pd
from io import StringIO
filas = []
for n in [1_000, 100_000, 1_000_000]:
    for hilos in [128, 256, 512]:
        salida = subprocess.check_output([str(binario), str(n), str(hilos), '15'], text=True)
        assert 'VALIDACION,OK' in salida, salida
        lineas = [linea for linea in salida.strip().splitlines() if linea.startswith('N,') or linea[:1].isdigit()]
        tabla = pd.read_csv(StringIO('\n'.join(lineas)))
        filas.append(tabla)
resultado = pd.concat(filas, ignore_index=True)
display(resultado)


## Guardar resultados y graficar

No se debe inventar ningún resultado. La tabla se genera cuando el hardware de Colab ejecute las pruebas.


In [ ]:
import matplotlib.pyplot as plt
resultados_dir = modulo / 'resultados'
resultados_dir.mkdir(exist_ok=True)
resultado.to_csv(resultados_dir / 'eta_experimento.csv', index=False)
for hilos, grupo in resultado.groupby('HILOS'):
    plt.plot(grupo['N'], grupo['SPEEDUP_total'], marker='o', label=f'{hilos} hilos')
plt.axhline(1, linestyle='--', linewidth=1, label='Empate CPU/GPU')
plt.xscale('log')
plt.xlabel('Registros (N)')
plt.ylabel('Speedup total (CPU/GPU total)')
plt.title('EquipeGo 2: ETA — comparación CPU vs GPU')
plt.legend()
plt.grid(True)
plt.show()
print('CSV local:', resultados_dir / 'eta_experimento.csv')


## Preguntas para justificar ante JJ

1. ¿Por qué cada hilo CUDA puede calcular una ETA de forma independiente?
2. ¿Por qué mantenemos Dijkstra en el proyecto original y paralelizamos solo la fase de cómputo por candidato?
3. ¿Por qué puede ganar la CPU cuando N es pequeño?
4. ¿Qué diferencia hay entre `GPU_kernel_ms` y `GPU_total_ms`?
5. ¿Qué cambia al utilizar 128, 256 y 512 hilos por bloque?
6. ¿Cómo influye `double` (FP64) en la GPU asignada?
7. ¿Por qué conviene registrar los detalles de CPU/GPU y repetir las pruebas?

**Importante:** no extrapolar el speedup de este microbenchmark al simulador completo de EquipeGo.
